In [7]:
import csv
import os
import pandas as pd
import glob
import sys

sys.path.append("../../src")

from util import * 

# make all-in-one table

In [2]:
tdc_paths = glob.glob('../../data/processed/tdc/*/*_date.tsv')
ml_paths = glob.glob('../../data/processed/moleculenet/moleculenet_patentdate/*/date_dataset.tsv')
ml_task_paths = glob.glob('../../data/processed/moleculenet/moleculenet_patentdate/*/tasks.txt')

In [3]:
print(len(tdc_paths), len(ml_paths), len(ml_task_paths))

45 24 24


In [4]:
output_path = '../../data/processed/all_date_datasets_for_tox_benchmark.tsv'
with open(output_path, 'a', newline='') as outfile:
    writer = csv.writer(outfile, delimiter='\t')
    writer.writerow(['smiles', 'year', 'YYMMDD', 'toxicity', 'dataset'])

    for path in tdc_paths:
        task_name = "tdc_" + path.split('/')[-2] + "_" + path.split('/')[-1].replace('_date.tsv', '')
        df = pd.read_csv(path, sep='\t')
        for i in range(len(df)):
            writer.writerow([df.iloc[i,0], df.iloc[i,2], convert_decimal_to_date(df.iloc[i,2]), df.iloc[i,1], task_name])

In [5]:
with open(output_path, 'a', newline='') as outfile:
    writer = csv.writer(outfile, delimiter='\t')

    for i in range(len(ml_paths)):
        task_names = []
        with open(ml_task_paths[i], 'r') as f:
            for line in f:
                task_names.append(line.strip())
        df = pd.read_csv(ml_paths[i], sep='\t', header=None)
        for j in range(len(df)):
            for k in range(len(task_names)):
                if pd.isna(df.iloc[j, k+1]):
                    continue
                task_name = "ml_" + ml_paths[i].split('/')[-2] + "_" + task_names[k]
                writer.writerow([df.iloc[j,0], df.iloc[j, 1], convert_decimal_to_date(df.iloc[j, 1]), df.iloc[j,k+2], task_name])

In [6]:
df = pd.read_csv(output_path, sep='\t')

In [7]:
df

,smiles,year,YYMMDD,toxicity,dataset
0,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,1973.121,19730213,0.0,tdc_HTS_3_HIV
1,O=C(O)c1ccccc1O,1973.063,19730123,0.0,tdc_HTS_3_HIV
2,O=[N+]([O-])c1ccc(SSc2ccc([N+](=O)[O-])cc2[N+]...,1982.145,19820222,0.0,tdc_HTS_3_HIV
3,O=[N+]([O-])c1ccccc1SSc1ccccc1[N+](=O)[O-],1976.036,19760113,0.0,tdc_HTS_3_HIV
4,CC(C)(CCC(=O)O)CCC(=O)O,1979.025,19790109,0.0,tdc_HTS_3_HIV
...,...,...,...,...,...
11255131,Cn1cnnc1SCC(=O)OCN1C(=O)c2ccccc2C1=O,2005.323,20050428,0.0,ml_muv_MUV-832
11255132,Cn1cnnc1SCC(=O)OCN1C(=O)c2ccccc2C1=O,2005.323,20050428,0.0,ml_muv_MUV-846
11255133,Cn1cnnc1SCC(=O)OCN1C(=O)c2ccccc2C1=O,2005.323,20050428,0.0,ml_muv_MUV-852
11255134,Cn1cnnc1SCC(=O)OCN1C(=O)c2ccccc2C1=O,2005.323,20050428,0.0,ml_muv_MUV-858


# update to huggingface
update too Large files to huggingface

In [ ]:
from huggingface_hub import upload_file

REPO_ID="mizuno-group/patent-time-split-toxicity-benchmark"
output_path = '../../data/processed/all_date_datasets_for_tox_benchmark.tsv'

upload_file(
    path_or_fileobj=output_path,
    path_in_repo="all_date_datasets_for_tox_benchmark.tsv",
    repo_id=REPO_ID,
    repo_type="dataset",
    create_pr=1
)


# analysis

In [2]:
output_path = '../../data/processed/all_date_datasets_for_tox_benchmark.tsv'
df = pd.read_csv(output_path, sep="\t")

In [3]:
df.head()

,smiles,year,YYMMDD,toxicity,dataset
0,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,1973.121,19730213,0.0,tdc_HTS_3_HIV
1,O=C(O)c1ccccc1O,1973.063,19730123,0.0,tdc_HTS_3_HIV
2,O=[N+]([O-])c1ccc(SSc2ccc([N+](=O)[O-])cc2[N+]...,1982.145,19820222,0.0,tdc_HTS_3_HIV
3,O=[N+]([O-])c1ccccc1SSc1ccccc1[N+](=O)[O-],1976.036,19760113,0.0,tdc_HTS_3_HIV
4,CC(C)(CCC(=O)O)CCC(=O)O,1979.025,19790109,0.0,tdc_HTS_3_HIV


In [4]:
print(len(set(df["smiles"])))
print(len(set(df["dataset"])))

80292
1517
